# RAG — RAGAS evaluation on benchmark CSV

**Goal:** Run [RAGAS](https://docs.ragas.io/) default metrics on rows produced by `scripts/run_rag_benchmark.py`:
- **faithfulness** — claims in the answer supported by retrieved contexts  
- **answer_relevancy** — answer related to the question  
- **context_precision** — precision of retrieved contexts w.r.t. the reference  
- **context_recall** — how much of the reference is covered by contexts  

**Prerequisite:** `benchmark_full_results.csv` must exist. Run from `rag_lab`:
`python scripts/run_rag_benchmark.py --evaluation-mode`  
For **meaningful file-level retrieval** in the CSV (and fairer contexts vs. a single-doc index), prefer **`--unified-index`** (same idea as notebook 07).

**Requirements:** `OPENAI_API_KEY` in `rag_lab/.env`; install deps (cell below). Uses **LangChain** `OpenAIEmbeddings` for RAGAS compatibility with ragas 0.4.x.

In [1]:
# Install once if needed:
# %pip install -q "ragas>=0.4.0" "datasets>=2.14.0"


## 1) Paths, load CSV, build HuggingFace `Dataset`

RAGAS v2-style columns: `user_input`, `response`, `retrieved_contexts` (list of strings per row), `reference` (ground truth).

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from ragas import evaluate
from ragas.llms import llm_factory

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

_SRC = ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

load_dotenv(ROOT / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in rag_lab/.env")

# Prefer the latest benchmark outputs if available.
candidate_csvs = [
    ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark" / "benchmark_full_results.csv",
    ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark_final_qdesign" / "benchmark_full_results.csv",
]
BENCHMARK_CSV = next((p for p in candidate_csvs if p.is_file()), None)
if BENCHMARK_CSV is None:
    raise FileNotFoundError(
        "Missing benchmark_full_results.csv in rag_benchmark or rag_benchmark_final_qdesign."
    )

OUT_DIR = ROOT / "data" / "outputs" / "evaluation" / "ragas"

# Limit rows for quick run in notebook execution.
# Set to None to evaluate all rows.
MAX_ROWS: int | None = 20

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = "text-embedding-3-small"

df = pd.read_csv(BENCHMARK_CSV)
if MAX_ROWS is not None:
    df = df.head(int(MAX_ROWS))


def parse_contexts(raw) -> list[str]:
    if pd.isna(raw) or raw == "":
        return []
    try:
        data = json.loads(raw)
    except Exception:
        return []
    if not isinstance(data, list):
        return []
    return [str(x).strip() for x in data if str(x).strip()]


payload = {
    "question_id": [],
    "user_input": [],
    "response": [],
    "retrieved_contexts": [],
    "reference": [],
}
for _, row in df.iterrows():
    question = str(row.get("question", "")).strip()
    response = str(row.get("generated_answer", "")).strip()
    reference = str(row.get("expected_answer", "")).strip()
    contexts = parse_contexts(row.get("retrieved_contexts_json", ""))

    # Keep only evaluable rows.
    if not question or not response or not reference or not contexts:
        continue

    payload["question_id"].append(str(row.get("question_id", "")))
    payload["user_input"].append(question)
    payload["response"].append(response)
    payload["retrieved_contexts"].append(contexts)
    payload["reference"].append(reference)

meta_ids = payload.pop("question_id")
hf_ds = Dataset.from_dict(payload)
print("ROOT:", ROOT)
print("Benchmark CSV:", BENCHMARK_CSV)
print("Rows loaded:", len(df))
print("Rows evaluable by RAGAS:", len(hf_ds))
print("Chat model:", CHAT_MODEL)

ROOT: D:\Qbrainpython\QBrain\rag_lab
Benchmark CSV: D:\Qbrainpython\QBrain\rag_lab\data\outputs\evaluation\rag_benchmark\benchmark_full_results.csv
Rows loaded: 20
Rows evaluable by RAGAS: 20
Chat model: gpt-4o-mini


## 2) Run `evaluate` (default metrics)



In [2]:
# Run RAGAS with default metrics (faithfulness, answer_relevancy, context_precision, context_recall)
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
ragas_llm = llm_factory(model=CHAT_MODEL, client=openai_client)
ragas_embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

result = evaluate(
    dataset=hf_ds,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

df_scores = result.to_pandas()
df_scores.insert(0, "question_id", meta_ids[: len(df_scores)])

csv_path = OUT_DIR / "ragas_default_metrics.csv"
json_path = OUT_DIR / "ragas_default_metrics.json"

df_scores.to_csv(csv_path, index=False)
with json_path.open("w", encoding="utf-8") as f:
    f.write(df_scores.to_json(orient="records", force_ascii=False, indent=2))

print("RAGAS rows:", len(df_scores))
print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)
print("Mean scores:")
print(df_scores.select_dtypes(include=["number"]).mean(numeric_only=True).round(4))

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

RAGAS rows: 20
Saved CSV: D:\Qbrainpython\QBrain\rag_lab\data\outputs\evaluation\ragas\ragas_default_metrics.csv
Saved JSON: D:\Qbrainpython\QBrain\rag_lab\data\outputs\evaluation\ragas\ragas_default_metrics.json
Mean scores:
answer_relevancy     0.8070
context_precision    0.8535
faithfulness         0.9583
context_recall       0.9000
dtype: float64


In [10]:
## 4) Compare RAGAS means with benchmark summary (optional)

benchmark_summary_csv = ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark" / "overall_summary.csv"
if benchmark_summary_csv.is_file():
    benchmark_df = pd.read_csv(benchmark_summary_csv)
    print("Benchmark overall summary:")
    display(benchmark_df)
else:
    print("No benchmark overall_summary.csv found at:")
    print(benchmark_summary_csv)

print("\nRAGAS mean scores:")
ragas_means = df_scores.select_dtypes(include=["number"]).mean(numeric_only=True).sort_index()
print(ragas_means.round(4))

Benchmark overall summary:


,metric,value
0,srs_files,5.000000
1,questions,60.000000
2,retrieval_hit@5,1.000000
3,retrieval_precision@5,0.200000
4,retrieval_recall@5,1.000000
5,retrieval_mrr,1.000000
6,generation_accuracy_sim>=0.65,0.783333
7,generation_avg_similarity,0.767639
8,e2e_success_rate,0.783333



RAGAS mean scores:
answer_relevancy     0.8070
context_precision    0.8535
context_recall       0.9000
faithfulness         0.9583
dtype: float64
